<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>LSTM과 Attention 메커니즘(LSTM & Attention Mechanism)</strong>에 대해 학습합니다.</br></br>
>LSTM의 게이트 구조와 Luong Attention의 컨텍스트 벡터 계산을 학습해봅시다.

</br>

# LSTM (Long Short-Term Memory)
> LSTM은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">게이트 메커니즘으로 장기 의존성 문제를 해결</mark>한 RNN 변형입니다.

RNN은 긴 문장을 기억하지 못합니다. "오늘 아침 서울에서 출발한 기차가 부산에 도착했다"에서 RNN이 "도착했다"를 처리할 때 "기차"의 정보는 이미 희미해져 있습니다. 역전파 시 기울기는 시간 축을 따라 반복 곱해지면서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기하급수적으로 소실</mark>되고, 시퀀스가 길수록 초기 정보가 사라지는 기울기 소실(Vanishing Gradient) 문제가 발생합니다.</br>

LSTM은 셀 상태(Cell State)라는 별도 통로로 이 문제를 해결합니다. 시그모이드 함수(출력 0~1)를 활용한 세 개의 게이트가 정보 흐름을 정밀하게 제어하는데, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Forget Gate</mark>는 과거 정보 중 버릴 것을, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Input Gate</mark>는 현재 입력에서 저장할 것을, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Output Gate</mark>는 셀 상태 중 출력할 것을 각각 원소별 곱(Element-wise Product)으로 결정합니다. 셀 상태는 덧셈 연산으로 갱신되므로 기울기가 소실 없이 먼 시점까지 전달됩니다.</br>

그럼에도 Seq2Seq+LSTM에는 정보 병목이 존재합니다. 인코더는 전체 입력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">하나의 고정 크기 벡터</mark>로 압축해야 하므로, 문장이 길어질수록 정보가 손실됩니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Attention 메커니즘</mark>은 디코더가 매 스텝마다 인코더의 모든 출력을 직접 참조하도록 허용하여 이 병목을 우회하며, 이후 Transformer의 핵심 구조로 발전하여 현대 LLM의 근간이 됩니다.

</br>

## LSTM 셀 구조

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">게이트</th>
      <th>역할</th>
      <th>수식</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Forget Gate ($f_t$)</td><td>이전 정보 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">삭제 비율</mark> 결정</td><td>$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$</td></tr>
    <tr><td style="text-align:center">Input Gate ($i_t$)</td><td>새 정보 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">저장 비율</mark> 결정</td><td>$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$</td></tr>
    <tr><td style="text-align:center">Output Gate ($o_t$)</td><td>셀 상태에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력 비율</mark> 결정</td><td>$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 1: 필요한 모듈을 불러오고, LSTM 레이어를 생성한 뒤, 랜덤 입력 텐서로 출력, 은닉 상태, 셀 상태의 shape를 출력해봅시다.

import torch
import torch.nn as nn

lstm = nn.LSTM(input_size=256, hidden_size=512, num_layers=2,
               batch_first=True, dropout=0.3)

embedded = torch.randn(4, 10, 256)    # (batch=4, seq=10, embed=256)
output, (hidden, cell) = lstm(embedded)

print(f"output shape: {output.shape}")    # 모든 시점 출력
print(f"hidden shape: {hidden.shape}")    # 은닉 상태
print(f"cell shape:   {cell.shape}")      # 셀 상태

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">출력</th>
      <th style="text-align:center">Shape</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">output</td><td style="text-align:center">(4, 10, 512)</td><td>모든 시점의 출력</td></tr>
    <tr><td style="text-align:center">hidden</td><td style="text-align:center">(2, 4, 512)</td><td>은닉 상태 (num_layers, batch, hidden)</td></tr>
    <tr><td style="text-align:center">cell</td><td style="text-align:center">(2, 4, 512)</td><td>셀 상태 (num_layers, batch, hidden)</td></tr>
  </tbody>
</table>

💡LSTM vs GRU
> GRU는 게이트가 2개(Reset, Update)로 LSTM보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">파라미터가 적고 빠릅니다</mark>.</br>
> 성능은 비슷한 경우가 많아 데이터 크기에 따라 선택합니다.

</br>

# Attention 메커니즘 (Attention Mechanism)
> 디코더가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 시퀀스의 관련 부분에 선택적으로 집중</mark>할 수 있게 하는 기법입니다.

</br>

## Luong Attention 구현

In [ ]:
# TODO 2: 인코더 출력과 디코더 출력을 랜덤 텐서로 생성하고, 행렬 곱으로 Attention Score를 계산한 뒤, 소프트맥스로 Attention Weights를 구하고, 가중합으로 Context Vector를 계산하여 출력해봅시다.

encoder_outputs = torch.randn(1, 5, 512)    # Encoder 전체 출력
decoder_output = torch.randn(1, 1, 512)      # Decoder 현재 출력

# 1. Attention Score
scores = torch.bmm(decoder_output, encoder_outputs.transpose(1, 2))
print(f"Attention Scores: {scores.shape}")
print(f"Score 값: {scores.data.round(decimals=2)}")

# 2. Attention Weights (Softmax)
attn_weights = torch.softmax(scores, dim=-1)
print(f"\nAttention Weights: {attn_weights.data.round(decimals=3)}")

# 3. Context Vector (가중합)
context = torch.bmm(attn_weights, encoder_outputs)
print(f"\nContext Vector: {context.shape}")

💡Attention이 왜 효과적인가?
> 기존 Seq2Seq는 전체 입력을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">하나의 고정 벡터로 압축</mark>해야 했습니다.</br>
> Attention은 각 디코딩 단계에서 입력의 관련 부분에 직접 접근하므로 정보 손실이 줄어듭니다.

💡Self-Attention과의 차이
> 여기서 배우는 Attention은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Cross-Attention</mark> (인코더↔디코더 간)입니다.</br>
> Transformer의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Self-Attention</mark>은 같은 시퀀스 내부에서 관계를 계산합니다.

</br>

## LuongAttention 클래스 구현

In [ ]:
# TODO 3: LuongAttention 클래스를 정의하여 순전파에서 Attention Score, Weights, Context Vector를 계산하고, 인스턴스를 생성하여 context와 weights의 shape를 출력해봅시다.

class LuongAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

    def forward(self, decoder_output, encoder_outputs):
        # decoder_output: (batch, 1, hidden)
        # encoder_outputs: (batch, src_len, hidden)
        scores = torch.bmm(decoder_output, encoder_outputs.transpose(1, 2))
        # scores: (batch, 1, src_len)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)
        # context: (batch, 1, hidden)
        return context, attn_weights

attn = LuongAttention(hidden_dim=512)
enc_out = torch.randn(1, 5, 512)    # (batch=1, src_len=5, hidden=512)
dec_out = torch.randn(1, 1, 512)    # (batch=1, 1, hidden=512)
ctx, weights = attn(dec_out, enc_out)
print(f"Context shape: {ctx.shape}")        # Context shape: torch.Size([1, 1, 512])
print(f"Weights shape: {weights.shape}")    # Weights shape: torch.Size([1, 1, 5])

</br>

## AttentionDecoder 패턴

In [ ]:
# TODO 4: AttentionDecoder 클래스를 정의하여 임베딩, LSTM, LuongAttention, 출력 선형 레이어를 초기화하고, 순전파에서 디코더 출력과 컨텍스트를 결합하여 예측과 attention weights를 반환한 뒤 테스트해봅시다.

class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.attention = LuongAttention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)   # context + output 결합

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)                              # (batch, 1, embed)
        dec_out, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        context, attn_weights = self.attention(dec_out, encoder_outputs)
        combined = torch.cat([dec_out, context], dim=-1)          # (batch, 1, hidden*2)
        prediction = self.fc(combined)                            # (batch, 1, vocab_size)
        return prediction, hidden, cell, attn_weights

dec = AttentionDecoder(vocab_size=100, embed_dim=256, hidden_dim=512)
enc_outputs = torch.randn(1, 5, 512)
h = torch.zeros(1, 1, 512)
c = torch.zeros(1, 1, 512)
token = torch.LongTensor([[2]])   # 시작 토큰
pred, h, c, w = dec(token, h, c, enc_outputs)
print(f"예측 shape: {pred.shape}")           # 예측 shape: torch.Size([1, 1, 100])
print(f"Attention weight shape: {w.shape}")  # Attention weight shape: torch.Size([1, 1, 5])

💡AttentionDecoder의 핵심
> 매 디코딩 스텝마다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어떤 인코더 출력에 집중할지</mark> 동적으로 결정합니다.</br>
> `fc` 레이어 입력이 `hidden_dim * 2`인 이유는 디코더 출력과 컨텍스트 벡터를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">concat</mark>하기 때문입니다.